# Khám phá dữ liệu Abalone

Notebook này dùng để phân tích dữ liệu ban đầu cho bài toán dự đoán `Rings` của bộ dữ liệu Abalone.

## 1. Chuẩn bị thư viện

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Đọc dữ liệu

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'abalone.csv'

columns = [
    'Sex',
    'Length',
    'Diameter',
    'Height',
    'WholeWeight',
    'ShuckedWeight',
    'VisceraWeight',
    'ShellWeight',
    'Rings',
]

df = pd.read_csv(DATA_PATH, header=None, names=columns)
df.head()

## 3. Thông tin tổng quan

In [ ]:
print(f'Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột')
display(df.info())
display(df.describe(include='all').T)

## 4. Kiểm tra giá trị thiếu và trùng lặp

In [ ]:
missing_summary = df.isna().sum().to_frame('missing_count')
missing_summary['missing_ratio'] = missing_summary['missing_count'] / len(df)
display(missing_summary)

print('Số dòng trùng lặp:', df.duplicated().sum())

## 5. Phân tích biến phân loại `Sex`

In [ ]:
sex_counts = df['Sex'].value_counts()
display(sex_counts)

ax = sns.countplot(data=df, x='Sex', order=sex_counts.index)
ax.set_title('Phân phối biến Sex')
plt.show()

## 6. Phân phối biến mục tiêu `Rings`

In [ ]:
display(df['Rings'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Rings'], kde=True, ax=axes[0])
axes[0].set_title('Histogram của Rings')
sns.boxplot(x=df['Rings'], ax=axes[1])
axes[1].set_title('Boxplot của Rings')
plt.tight_layout()
plt.show()

## 7. Phân tích các biến số

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols].hist(bins=20, figsize=(16, 12))
plt.suptitle('Phân phối các biến số', y=1.02)
plt.tight_layout()
plt.show()

## 8. Tương quan giữa các biến số

In [ ]:
corr = df[numeric_cols].corr()
display(corr['Rings'].sort_values(ascending=False))

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Ma trận tương quan')
plt.show()

## 9. Quan hệ giữa đặc trưng và biến mục tiêu

In [ ]:
key_features = ['Length', 'Diameter', 'Height', 'WholeWeight', 'ShellWeight']

fig, axes = plt.subplots(len(key_features), 1, figsize=(8, 4 * len(key_features)))
for ax, col in zip(axes, key_features):
    sns.scatterplot(data=df, x=col, y='Rings', hue='Sex', alpha=0.6, ax=ax)
    ax.set_title(f'{col} và Rings')
plt.tight_layout()
plt.show()

## 10. Kiểm tra ngoại lệ

In [ ]:
fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(10, 4 * len(numeric_cols)))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(f'Boxplot của {col}')
plt.tight_layout()
plt.show()

## 11. Nhận xét ban đầu

In [ ]:
print('1. Dữ liệu không có giá trị thiếu.')
print('2. Bài toán phù hợp với hồi quy, biến mục tiêu là Rings.')
print('3. Sex là biến phân loại cần mã hóa ở bước tiền xử lý.')
print('4. Cần kiểm tra kỹ ngoại lệ và mức độ lệch phân phối của các biến trọng lượng.')
print('5. Nên thử cả mô hình tuyến tính và mô hình cây/ensemble ở giai đoạn baseline.')